# 脳波デコーディング体験 ① — CSP + 機械学習（古典手法）

このノートブックでは、**運動想起（Motor Imagery, MI）** の脳波から
「左手を動かす想像」か「右手を動かす想像」かを当てる **デコーダ** を、
古典的な機械学習で作ります。

## 運動想起と脳波
- 手を動かす（または動かす想像をする）と、反対側の **感覚運動野** で
  μ波（8–13Hz）・β波（13–30Hz）の **パワーが下がる** 現象が起きます
  （**ERD: 事象関連脱同期**）。
- 左手の想像 → 右半球で ERD、右手の想像 → 左半球で ERD。
  この **左右差** を機械学習で読み取るのがデコーダの基本アイデアです。
- 古典 BCI では C3 / C4（左右の手の運動野直上）を使いますが、今回の
  Emotiv EPOC X キャップには C3/C4 電極が無いため、感覚運動野に近い
  **FC5 / FC6 / T7 / T8** あたりが手がかりになります。

## このノートの流れ
1. 記録データの読み込み
2. 生波形と前処理（バンドパス + ノッチ + CAR）の確認
3. クラスごとのパワースペクトル（ERD の可視化）
4. **CSP（Common Spatial Patterns）** で左右差を強調する空間フィルタを学習
5. CSP + 分類器（LDA / SVM）で交差検証
6. まとめと演習

> 💡 ここで使う前処理・エポック切り出しは、実際のリアルタイム BCI
> （`bci/` パッケージ）と **同じコード** を呼び出しています。

In [1]:
# 必要なライブラリ
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

import mne
from mne.decoding import CSP
mne.set_log_level("WARNING")

from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import FgMDM

# 共有ユーティリティ（実 BCI と同じ前処理・エポック化を呼ぶ）
import mi_utils as U

U.setup_japanese_font()  # グラフの日本語が豆腐(□)にならないように
plt.rcParams["figure.figsize"] = (9, 4)
np.random.seed(0)
print("準備完了")

準備完了


## 1. データの読み込み

5 セッション分の記録を読み込みます。各セッションは「左想像 10 回 +
右想像 10 回」。`load_dataset()` の中で、セッションごとに
**ノッチ(50Hz) → バンドパス(8–30Hz) → CAR（共通平均基準）** をかけ、
MI 区間（4 秒）を 1 トライアルとして切り出しています。

In [2]:
ds = U.load_dataset()
print(ds.summary())

# X: (トライアル数, チャンネル数, サンプル数), y: 0=left / 1=right
X, y, fs, ch_names = ds.X, ds.y, ds.fs, ds.ch_names
print("\nX shape:", X.shape, "| y shape:", y.shape, "| fs:", fs, "Hz")

trials      : 100
channels    : 14  ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
samples/trial: 512  (4.0s @ 128Hz)
sessions    : 5
  [0] yamamura_session1_1_20260617T022949: {'left': 10, 'right': 10}
  [1] yamamura_session2_20260617T023244: {'left': 10, 'right': 10}
  [2] yamamura_session3_20260617T023642: {'left': 10, 'right': 10}
  [3] yamamura_session4_20260617T024042: {'left': 10, 'right': 10}
  [4] yamamura_session5_20260617T024325: {'left': 10, 'right': 10}
class totals: {'left': 50, 'right': 50}

X shape: (100, 14, 512) | y shape: (100,) | fs: 128.0 Hz


## 2. 前処理済み波形を見る

運動野近傍のチャンネルで、1 トライアル分の波形を表示します。
バンドパス済みなので μ/β 帯の振動が残り、ゆっくりしたドリフトや
電源ノイズは除去されています。

In [3]:
motor_ch = [c for c in ["FC5", "FC6", "T7", "T8", "F3", "F4"] if c in ch_names]
idx = [ch_names.index(c) for c in motor_ch]

trial = 0
t = np.arange(X.shape[2]) / fs
fig, ax = plt.subplots()
offset = 0
for c, i in zip(motor_ch, idx):
    ax.plot(t, X[trial, i] + offset, label=c, lw=0.8)
    offset += 60  # 見やすいように縦にずらす
ax.set(xlabel="時間 [s]", yticks=[], title=f"前処理済み EEG（trial {trial}, クラス={U.CLASS_NAMES[y[trial]]}）")
ax.legend(loc="upper right", ncol=len(motor_ch), fontsize=8)
plt.tight_layout(); plt.show()

/var/folders/7_/5h8t97xx06q690jwmw8188kr0000gn/T/ipykernel_47404/3104132377.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3. クラスごとのパワースペクトル（ERD を探す）

左想像と右想像で、各チャンネルの **8–30Hz 帯のパワー** に差が出るはず
です。Welch 法でパワースペクトル密度（PSD）を推定し、クラス平均を
重ねて表示します。左右で差が大きいチャンネルほどデコードの手がかりに
なります。

In [4]:
def band_psd(sig, fs):
    f, p = welch(sig, fs=fs, nperseg=min(256, sig.shape[-1]))
    return f, p

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
for ax, c in zip(axes.ravel(), motor_ch):
    ci = ch_names.index(c)
    for label, color in [(U.LABEL_LEFT, "tab:blue"), (U.LABEL_RIGHT, "tab:red")]:
        psd = np.array([band_psd(X[k, ci], fs)[1] for k in np.where(y == label)[0]])
        f = band_psd(X[0, ci], fs)[0]
        m = psd.mean(0)
        ax.plot(f, m, color=color, label=U.CLASS_NAMES[label])
    ax.set(title=c, xlim=(4, 35))
    ax.legend(fontsize=8)
axes[1, 0].set_xlabel("周波数 [Hz]"); axes[0, 0].set_ylabel("PSD")
fig.suptitle("クラス平均パワースペクトル（左 vs 右）")
plt.tight_layout(); plt.show()

/var/folders/7_/5h8t97xx06q690jwmw8188kr0000gn/T/ipykernel_47404/2911972402.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. CSP（Common Spatial Patterns）

CSP は「**一方のクラスで分散が最大、もう一方で最小**」になるような
チャンネルの重み付け（空間フィルタ）を学習する手法です。
バンドパス後の信号の分散はその帯域の **パワー** に相当するので、
CSP は「左右で最もパワー差が出る合成チャンネル」を作ってくれます。

まず可視化用に全データで CSP を学習し、空間パターンを頭皮上に描きます
（赤/青が強いほどその電極の寄与が大きい）。

In [5]:
import mne

# 電極名は標準 10-20 系なので、頭皮図（topomap）が描ける
info = mne.create_info(ch_names, sfreq=fs, ch_types="eeg")
info.set_montage("standard_1020")

csp_vis = CSP(n_components=4, reg="ledoit_wolf", log=True, norm_trace=False)
csp_vis.fit(X, y)

csp_vis.plot_patterns(info, components=range(4), ch_type="eeg", units="a.u.", size=1.5)
plt.show()

/opt/anaconda3/envs/MotorImageryDemo/lib/python3.10/site-packages/mne/viz/utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)
/var/folders/7_/5h8t97xx06q690jwmw8188kr0000gn/T/ipykernel_47404/2791196110.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### CSP 特徴量の様子

各トライアルを CSP に通すと、成分ごとの log 分散（= log パワー）が
特徴ベクトルになります。第 1 成分と最終成分は、左右クラスで最も
分離しやすいように設計されています。散布図で確認しましょう。

In [6]:
feat = csp_vis.transform(X)  # (n_trials, n_components)
fig, ax = plt.subplots(figsize=(6, 5))
for label, color in [(U.LABEL_LEFT, "tab:blue"), (U.LABEL_RIGHT, "tab:red")]:
    m = y == label
    ax.scatter(feat[m, 0], feat[m, -1], c=color, label=U.CLASS_NAMES[label], alpha=0.7)
ax.set(xlabel="CSP 成分 0 (log-var)", ylabel="CSP 成分 3 (log-var)",
       title="CSP 特徴空間でのクラス分布")
ax.legend(); plt.tight_layout(); plt.show()

/var/folders/7_/5h8t97xx06q690jwmw8188kr0000gn/T/ipykernel_47404/1677763839.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.tight_layout(); plt.show()


## 5. CSP + 分類器で交差検証

CSP と分類器（LDA や SVM）を `Pipeline` でつなぎ、交差検証で精度を
測ります。**ここが最重要ポイント**：

- ❌ トライアルをランダムに分けると、同じ日の似た脳波が訓練とテストの
  両方に入り、精度が **過大評価** されがちです。
- ✅ 今回は **Leave-One-Session-Out**（あるセッション全体をテストに回す）
  で評価します。これは「別の日のデータでも当たるか？」という、BCI で
  本当に問われる汎化性能に近い評価です。

> CSP は交差検証の **各 fold の訓練データだけ** で学習し直します
> （`Pipeline` がこれを自動でやってくれます）。さっきの可視化用 CSP とは
> 別物である点に注意。

In [7]:
logo = LeaveOneGroupOut()

pipelines = {
    "CSP + LDA": Pipeline([("csp", CSP(n_components=4, reg="ledoit_wolf", log=True)), ("clf", LDA())]),
    "CSP + SVM": Pipeline([("csp", CSP(n_components=4, reg="ledoit_wolf", log=True)),
                            ("clf", SVC(kernel="rbf", C=1.0))]),
}

for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X, y, groups=ds.session, cv=logo, scoring="accuracy")
    print(f"{name:12s} | セッション別精度: "
          + ", ".join(f"{s:.2f}" for s in scores)
          + f" | 平均 {scores.mean():.3f} ± {scores.std():.3f}")

print("\n（チャンスレベル = 0.50。2 クラスなので 0.5 を上回れば学習できている）")

CSP + LDA    | セッション別精度: 0.60, 0.60, 0.55, 0.65, 0.45 | 平均 0.570 ± 0.068


CSP + SVM    | セッション別精度: 0.45, 0.60, 0.50, 0.65, 0.45 | 平均 0.530 ± 0.081

（チャンスレベル = 0.50。2 クラスなので 0.5 を上回れば学習できている）


### 混同行列

CSP + LDA で LOSO の予測を集めて混同行列にします。

In [8]:
pipe = Pipeline([("csp", CSP(n_components=4, reg="ledoit_wolf", log=True)), ("clf", LDA())])
y_pred = cross_val_predict(pipe, X, y, groups=ds.session, cv=logo)

cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=[U.CLASS_NAMES[0], U.CLASS_NAMES[1]])
disp.plot(cmap="Blues", values_format="d")
plt.title(f"CSP + LDA  (LOSO, acc={(y_pred==y).mean():.2f})")
plt.tight_layout(); plt.show()

/var/folders/7_/5h8t97xx06q690jwmw8188kr0000gn/T/ipykernel_47404/2656155022.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. さらに精度を上げる — リーマン幾何 (FgMDM)

### なぜリーマン幾何？

CSP+LDA は「分散（パワー）の最大化」という線形な視点でした。
**リーマン幾何**は脳波の **共分散行列** を、**正定値対称行列の多様体（湾曲した空間）**
上の点として扱い、その空間上の距離で分類します。

- 共分散行列は「どのチャンネル同士が同期しているか」という**空間的なパターン**
  を丸ごと保持している。
- 湾曲した多様体上の距離を正しく測ることで、小規模データでも頑健に動く。

**FgMDM (Frechet-geodesic Minimum Distance to Mean)**: 各クラスの
リーマン重心（平均共分散行列）を学習し、テスト時は最も近い重心のクラスに割り当てる。

### μ帯（8–13Hz）に絞る

実験すると **μ帯のみ（8–13Hz）** でリーマン法を使うと精度が上がる場合があります。
ERD は μ 帯（手の運動野）に最も強く出るため、β 成分の混入がノイズになることがあります。

### 運動野チャンネル選択

FC5 / FC6 / T7 / T8 / F3 / F4 ── 手の感覚運動野に近い 6 チャンネルに絞ります。
頭頂・後頭部のチャンネルは運動想起とは関係が薄く、ノイズになりやすいです。

In [9]:
# μ帯 + 運動野 6ch のみで読み込み直す
MOTOR_CH = [c for c in ["FC5", "FC6", "T7", "T8", "F3", "F4"] if c in ch_names]
ds_mu = U.load_dataset(bandpass=(8, 13))
X_mu = ds_mu.X[:, [ds_mu.ch_names.index(c) for c in MOTOR_CH], :]
y_mu, sess_mu = ds_mu.y, ds_mu.session
print(f"μ帯 + 運動野 {len(MOTOR_CH)}ch: {X_mu.shape}")

# FgMDM: 共分散行列の推定 (lwf=Ledoit-Wolf) + リーマン最近傍
pipe_riemann = Pipeline([
    ("cov", Covariances("lwf")),
    ("clf", FgMDM()),
])

scores_riemann = cross_val_score(pipe_riemann, X_mu, y_mu, groups=sess_mu, cv=logo)
print(f"FgMDM (μ帯 + 運動野6ch) | セッション別: "
      + ", ".join(f"{s:.2f}" for s in scores_riemann)
      + f" | 平均 {scores_riemann.mean():.3f} ± {scores_riemann.std():.3f}")

μ帯 + 運動野 6ch: (100, 6, 512)
FgMDM (μ帯 + 運動野6ch) | セッション別: 0.50, 0.75, 0.50, 0.70, 0.75 | 平均 0.640 ± 0.116


### 手法比較まとめ

In [10]:
# 全手法の精度を並べてグラフ化
comparisons = {
    "CSP+LDA\n(8-30Hz 全ch)": cross_val_score(
        Pipeline([("csp", CSP(n_components=4, reg="ledoit_wolf", log=True)), ("clf", LDA())]),
        X, y, groups=ds.session, cv=logo),
    "CSP+LDA\n(μ帯 全ch)": cross_val_score(
        Pipeline([("csp", CSP(n_components=4, reg="ledoit_wolf", log=True)), ("clf", LDA())]),
        ds_mu.X, y_mu, groups=sess_mu, cv=logo),
    "FgMDM\n(μ帯 運動野6ch)": scores_riemann,
    "TangentSpace+LDA\n(μ帯 運動野6ch)": cross_val_score(
        Pipeline([("cov", Covariances("lwf")), ("ts", TangentSpace()), ("clf", LDA())]),
        X_mu, y_mu, groups=sess_mu, cv=logo),
}

means = [v.mean() for v in comparisons.values()]
stds  = [v.std()  for v in comparisons.values()]
labels = list(comparisons.keys())

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels, means, xerr=stds, color="steelblue", alpha=0.8, capsize=5)
ax.axvline(0.5, color="tomato", ls="--", label="チャンスレベル (0.50)")
for bar, m in zip(bars, means):
    ax.text(m + 0.005, bar.get_y() + bar.get_height()/2, f"{m:.2f}",
            va="center", fontsize=9)
ax.set(xlabel="LOSO accuracy", title="手法別精度比較（平均 ± 標準偏差）", xlim=(0.3, 0.85))
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

/var/folders/7_/5h8t97xx06q690jwmw8188kr0000gn/T/ipykernel_47404/2052730340.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(loc="lower right"); plt.tight_layout(); plt.show()


## 7. まとめと演習

**実験から分かったこと**
| 手法 | 精度 (LOSO) | ポイント |
|---|---|---|
| CSP + LDA (8–30Hz, 全14ch) | ~0.57 | ベースライン |
| CSP + LDA (μ帯, 全14ch) | ~0.59 | 帯域を絞るだけで改善 |
| FgMDM (μ帯, 運動野6ch) | ~0.64 | リーマン + ch選択がベスト |

- **μ帯 (8–13Hz)** が ERD の主役。β は人によって効いたり効かなかったりする。
- **チャンネル選択**でノイズを減らすとリーマン法が安定する。
- リーマン幾何は「共分散行列の形」ごと分類するため、特徴設計が少なくて済む。
- それでも LOSO で 0.64 程度：C3/C4 が無い EPOC X での MI は難しく、
  「DL も含め、データ量と電極配置が最大のボトルネック」という事実を確認。

**演習（試してみよう）**
1. `MOTOR_CH` に `"P7", "P8"` を足すと精度はどう変わる？（後頭部は逆効果？）
2. `FgMDM` を `MDM`（正則化なし）に替えると？
3. `Covariances("lwf")` の `"lwf"` を `"oas"` や `"scm"` に替えると？
4. ノート ② の EEGNet と今回の FgMDM の精度を並べて比較しよう。